# Build the parish site list

Create the parish directory dataset used by `02_build_page_list.ipynb`. This first pipeline step reads the selected dioceses, downloads their parish directories, and combines directory entries into one CSV. It does not visit individual parish websites or extract ministries.

Run the cells from top to bottom with the project root as the working directory. Input and output paths are relative to that directory.


## 0 Environment and imports

Use Python 3.10 or newer and install the project dependencies in the notebook's environment with `%pip install -e .` if needed. Scraping uses pandas, Requests, and Beautiful Soup through `utils.scraping.scrape_parishes`. This notebook does not use an LLM or require an API key.

Directory downloads use the shared HTML cache. `HTML_CACHE_DIR` in the environment takes precedence over the project-root `.env` setting; the default is `data/cache/html`. Successful responses are saved as URL-hashed `.html` files with `.url` files recording the final URL after redirects. Reruns reuse existing cached responses without refreshing them, preserving downloaded source data.


In [1]:
from pathlib import Path
import pandas as pd
from utils.scraping import scrape_parishes

## 1 Load and select dioceses

Read `data/01_build_site_list/diocese_site_info.csv`. The notebook uses these input columns:

| Column | Purpose |
| --- | --- |
| `diocese_name` | Diocese label attached to each scraped entry. |
| `parish_list_url` | URL of the diocese's parish directory. Required for selected dioceses. |
| `is_target` | Boolean selection flag; only rows marked `TRUE` are processed. |

Other input columns are retained in `dioceses_df` but are not exported to the parish list. Selection uses `is_target`; it does not separately filter on `is_ecatholic`. Selected directories must use the eCatholic markup supported by the scraper (`li.site .siteInfo`).

Review the displayed diocese names and directory URLs before continuing. Select at least one diocese and supply a valid directory URL for every selected row. Missing directory URLs raise a `ValueError` before scraping begins.


In [2]:
data_folder = Path("data/01_build_site_list")
input_path = Path(data_folder, "diocese_site_info.csv")

dioceses_df = pd.read_csv(input_path)
selected_dioceses = dioceses_df[dioceses_df["is_target"]]

if selected_dioceses["parish_list_url"].isna().any():
    raise ValueError("Expected eCatholic dioceses with parish directory URLs.")
selected_dioceses[["diocese_name", "parish_list_url"]]


,diocese_name,parish_list_url
0,Archdiocese of Mobile,https://mobarch.org/parishfinder
1,Diocese of Birmingham,https://www.bhmdiocese.org/parishfinder
2,Archdiocese of Anchorage-Juneau,https://www.aoaj.org/parishfinder
3,Diocese of Fairbanks,https://dioceseoffairbanks.org/parishfinder
6,Diocese of Tucson,https://diocesetucson.org/parishfinder
12,Diocese of Fresno,https://dioceseoffresno.org/parishfinder
13,Diocese of Monterey,https://www.dioceseofmonterey.org/parishfinder
20,Diocese of Santa Rosa,https://srdiocese.org/parishfinder
33,Diocese of Pensacola-Tallahassee,https://www.ptdiocese.org/parishfinder
49,Archdiocese of Indianapolis,https://www.archindy.org/parishfinder


## 2 Scrape directories and export the parish list

For each selected diocese, `scrape_parishes` reads its cached directory or downloads it with a 30-second request timeout. It extracts every matching directory entry, resolves relative website links against the directory's final URL, and prints the number of entries found. It reads server-rendered HTML and does not execute JavaScript.

The scraper removes exact duplicate rows within each directory. The notebook then combines the results, removes exact duplicates across the combined dataset, and writes `data/01_build_site_list/2_parish_list.csv` as UTF-8 without a DataFrame index. Rerunning this cell replaces the generated CSV. The `data/01_build_site_list/` directory must already exist, as it contains the input CSV.

The output contains these columns, in order:

| Column | Description |
| --- | --- |
| `name` | Directory entry name; required. |
| `address` | Address text shown in the directory, or an empty string. |
| `site_url` | Resolved website URL, or an empty string if no website link is listed. |
| `diocese` | Diocese name from the input file. |
| `category` | Group heading from the source directory, or an empty string; this is not a predicted ministry category. |
| `parish_list_url` | Original directory URL used to collect the entry. |

Entries are kept as listed by the directory, including any non-parish institutions. Duplicate removal compares all columns; entries with the same website but different directory details remain separate.


In [3]:
parish_frames = []
for row in selected_dioceses.itertuples(index=False):
    parish_df = scrape_parishes(str(row.diocese_name), str(row.parish_list_url))
    parish_frames.append(parish_df)
    print(f"{row.diocese_name}: {len(parish_df)} entries")

parishes_df = pd.concat(parish_frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
output_path = Path(data_folder, "2_parish_list.csv")
parishes_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"\nSaved {len(parishes_df)} entries to {output_path}")
parishes_df.head()

Archdiocese of Mobile: 86 entries
Diocese of Birmingham: 80 entries
Archdiocese of Anchorage-Juneau: 53 entries
Diocese of Fairbanks: 47 entries
Diocese of Tucson: 85 entries
Diocese of Fresno: 136 entries
Diocese of Monterey: 47 entries
Diocese of Santa Rosa: 61 entries
Diocese of Pensacola-Tallahassee: 57 entries
Archdiocese of Indianapolis: 132 entries

Saved 784 entries to data\01_build_site_list\2_parish_list.csv


,name,address,site_url,diocese,category,parish_list_url
0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",https://fatherpurcell.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
1,Holy Spirit Catholic Parish,"Montgomery, AL",https://holyspiritmgm.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",https://francisxseelos.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
3,Chapel of Our Lady of Bon Secour,"17266 County Road 49 South, Bon Secour, AL, 36511",https://www.stjohnms.com/our-lady-of-bon-secour,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
4,Christ the King Catholic Church,"711 College Avenue, Daphne, AL, 36526",https://ctkdaphne.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder


## 3 Review and continue

Check the per-diocese counts, total exported count, and sample rows above. Directory entries without a website remain in the output; the next notebook needs usable `site_url` values to discover parish pages.

HTTP, connection, and parsing failures stop the scraping cell. A missing entry name or a directory with no matching entries raises a `ValueError`; check the directory URL and whether its markup is supported. If the cell stops, the final CSV has not been rewritten by that run and any previous export may still be present.

Once the export completes, continue with `02_build_page_list.ipynb`, which reads `data/01_build_site_list/2_parish_list.csv`.
